# Workshop 03: The Forecaster\x27s Toolbox — Benchmarks, Residual Forensics & Prediction Intervals
**Machine Learning II: Forecasting Economic Time Series** — *Lazarski University*  
*Author:* Leonardo Di Caterina (`l.dicaterina@lazarski.pl`)  
*Theoretical Reference:* Hyndman & Athanasopoulos (2021), *Forecasting: Principles and Practice* (FPP3), Chapter 5.

---

### Workshop Objectives
1. **Implement 4 Vectorized Benchmark Forecasters**: Mean, Naïve, Seasonal Naïve, and Drift methods with strict temporal index alignment.
2. **Compute One-Step In-Sample Residuals & Table 5.2 Multi-Step Standard Errors**: Understand parameter degrees of freedom ($K$) and horizon scaling $\sigma\sqrt{h}$.
3. **Quantify Prediction Uncertainty**: Build analytical Gaussian intervals and non-parametric bootstrapped residual sample path ensembles.
4. **Automate Residual Diagnostic Auditing (`check_residuals`)**: Verify unbiasedness, Bartlett ACF bounds ($\pm 1.96/\sqrt{T}$), and the Ljung-Box portmanteau test.
5. **Backtest Empirical Coverage**: Audit whether nominal 95% intervals achieve true 95% empirical calibration across rolling windows.
6. **Implement Decomposition Forecasting (STLF)**: Decompose non-stationary seasonal series with STL, extrapolate components, and reseasonalise.


In [ ]:
# =============================================================================
# Global Imports & Reproducibility Setup
# =============================================================================
import os
import sys
from pathlib import Path
from typing import Dict, List, Any, Union, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import STL

# Enforce deterministic reproducibility (RULES.md §3)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Matplotlib styling for high-legibility econometric diagnostics
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 11

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
print("Workshop 03 environment initialized. Data directory:", DATA_DIR.resolve())


---
## Part 1: Foundational Benchmark Forecasting Engines (35 Minutes)

### Theoretical Foundation (FPP3 §5.2)
Before building complex econometric or deep learning models, we must establish baseline hurdles. If a complex algorithm cannot outperform these simple heuristics, it destroys economic value:

1. **Mean Method**: $\hat{y}_{T+h|T} = \bar{y} = \frac{1}{T}\sum_{t=1}^T y_t$. Optimal under squared error loss for stationary white noise around a fixed mean.
2. **Naïve Method**: $\hat{y}_{T+h|T} = y_T$. Sets all future forecasts to the most recently observed value; optimal for a pure random walk ($y_t = y_{t-1} + \varepsilon_t$).
3. **Seasonal Naïve Method**: $\hat{y}_{T+h|T} = y_{T+h - m(k+1)}$, where $m$ is seasonal period and $k = \lfloor(h-1)/m\rfloor$. Repeats the last observed seasonal cycle.
4. **Drift Method**: $\hat{y}_{T+h|T} = y_T + h\left(\frac{y_T - y_1}{T - 1}\right)$. Equivalent to drawing a line between the first and last historical observations and extrapolating it.


In [ ]:
def load_and_index_series(
    filepath: str,
    date_col: str,
    val_col: str,
    freq: Optional[str] = None
) -> pd.Series:
    """
    Ingests a raw time series CSV, enforces an immutable DatetimeIndex and calendar frequency.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    # same recipe as workshop_01 + _02 -> only new thing: freq can be None now (= "pandas, guess it yourself")

    df = pd.read_csv(filepath)
    # read the csv -> DataFrame (= a table) with one col per csv col (here: the date col + the value col)

    df[date_col] = pd.to_datetime(df[date_col])
    # in the csv the dates are just text like "1992-01-01" -> to_datetime turns them into REAL dates
    # real dates = pandas knows what a month / quarter / weekday is

    df = df.set_index(date_col).sort_index()
    # set_index -> the date col becomes the row labels (= the index) -> a DatetimeIndex
    # sort_index -> oldest date at the top, newest at the bottom (autograder checks monotonic increasing!)

    series = df[val_col].astype(float)
    # df[val_col] -> pick ONE col -> pd.Series (= a single col of the df, the dates stay as its index)
    # astype(float) -> make every value a decimal number (no ints / text) so the math later never breaks

    if freq is None:
        freq = pd.infer_freq(series.index)
    # no freq given -> pandas looks at the gaps between the dates and guesses (e.g. "MS" if all dates are the 1st of a month)
    # Note: infer_freq gives back None if it cant guess -> then we just skip the asfreq step below

    if freq is not None:
        series = series.asfreq(freq)
    # asfreq -> tells pandas the calendar step between two rows
    # "MS" = month start, "QS" = quarter start, "B" = business day (Mon-Fri)
    # if a date is MISSING in the csv, asfreq adds it as NaN -> gaps become visible instead of silently hidden

    series.name = val_col
    # keep the original col name as the series name (autograder checks this!)

    return series

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


def _generate_future_index(y_train: pd.Series, h: int) -> pd.DatetimeIndex:
    """Generates a DatetimeIndex of length h extending beyond the historical sample."""
    freq = y_train.index.freq
    if freq is None:
        inferred = pd.infer_freq(y_train.index)
        freq = inferred if inferred is not None else "D"
    
    last_date = y_train.index[-1]
    full_range = pd.date_range(start=last_date, periods=h + 1, freq=freq)
    return full_range[1:]


def forecast_mean(y_train: pd.Series, h: int) -> pd.Series:
    """Extrapolates sample mean horizontally across future horizons h >= 1."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    y_bar = y_train.dropna().mean()
    # dropna -> throw away missing values (NaN) so they dont break the average
    # y_bar = (y_1 + y_2 + ... + y_T) / T = the average of the WHOLE history

    future_idx = _generate_future_index(y_train, h)
    # the h dates right AFTER the last training date (helper above does the calendar work)

    return pd.Series(np.full(h, y_bar), index=future_idx, name="mean_forecast")
    # np.full(h, y_bar) -> array with h copies of the same number, e.g. [25., 25., 25.]
    # -> a flat horizontal line into the future

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


def forecast_naive(y_train: pd.Series, h: int) -> pd.Series:
    """Sets all future forecasts to the final observed value y_T."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    y_T = y_train.dropna().iloc[-1]
    # .iloc[-1] -> the LAST value by POSITION (iloc = "integer location", ignores the dates)
    # y_T = the most recent thing we know

    future_idx = _generate_future_index(y_train, h)

    return pd.Series(np.full(h, y_T), index=future_idx, name="naive_forecast")
    # every future step = y_T -> "tomorrow will look exactly like today" (best guess for a random walk e.g. stock prices)

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


def forecast_snaive(y_train: pd.Series, h: int, m: int) -> pd.Series:
    """Sets each forecast to the last value from the identical season."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    last_season = y_train.dropna().iloc[-m:].values
    # .iloc[-m:] -> the LAST m values = the most recent full seasonal cycle
    # e.g. quarterly beer, m = 4 -> last Q1, Q2, Q3, Q4
    # .values -> plain numpy array without the dates (we only need the numbers here)

    steps = np.arange(h)
    # np.arange(h) -> [0, 1, 2, ..., h-1] (python starts counting at 0!)

    fc_values = last_season[steps % m]
    # % = modulo = the remainder of a division: 0 % 4 = 0, 1 % 4 = 1, 3 % 4 = 3, 4 % 4 = 0, 5 % 4 = 1 ...
    # -> positions 0,1,2,3,0,1,2,3,... -> the last cycle gets repeated again and again
    # this is the formula y_{T+h-m(k+1)} from the markdown, just written as "which position in the last cycle"
    # e.g. autograder: [10, 20, 30, 40] with m = 2 -> last_season = [30, 40] -> h = 3 gives [30, 40, 30]

    future_idx = _generate_future_index(y_train, h)

    return pd.Series(fc_values, index=future_idx, name="snaive_forecast")

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


def forecast_drift(y_train: pd.Series, h: int) -> pd.Series:
    """Linearly extrapolates the historical boundary slope from y_1 to y_T."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    y = y_train.dropna()
    T = len(y)
    # T = number of observations we have

    slope = (y.iloc[-1] - y.iloc[0]) / (T - 1)
    # slope (the "drift") = (y_T - y_1) / (T - 1)
    # = total change from the FIRST to the LAST point, divided by the number of steps in between
    # = the average change per period
    # e.g. [10, 20, 30, 40] -> (40 - 10) / 3 = 10 per period

    steps = np.arange(1, h + 1)
    # h = 1, 2, ..., h (here we start at 1 bc one step ahead already gets 1x the slope)

    fc_values = y.iloc[-1] + steps * slope
    # y_hat_{T+h} = y_T + h * slope -> start at the last value and keep walking along the line
    # numpy does this for ALL steps at once (no for-loop needed = vectorized): 40 + [1, 2, 3] * 10 = [50, 60, 70]

    future_idx = _generate_future_index(y_train, h)

    return pd.Series(fc_values, index=future_idx, name="drift_forecast")

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstration: Benchmark Showdown on Australian Quarterly Beer (FPP3 Figure 5.7)
beer = load_and_index_series(str(DATA_DIR / "aus_beer.csv"), "observation_date", "beer_production", freq="QS")
train_beer = beer.loc[:"2006-10-01"]
test_beer = beer.loc["2007-01-01":]
h_beer = len(test_beer)

fc_mean = forecast_mean(train_beer, h=h_beer)
fc_naive = forecast_naive(train_beer, h=h_beer)
fc_snaive = forecast_snaive(train_beer, h=h_beer, m=4)
fc_drift = forecast_drift(train_beer, h=h_beer)

plt.figure(figsize=(12, 5))
plt.plot(train_beer.index[-32:], train_beer.iloc[-32:], label="Historical Training", color="#1A3A6B", lw=2)
plt.plot(test_beer.index, test_beer.values, label="Actual Test Data", color="black", lw=2.5, ls="--")
plt.plot(fc_mean.index, fc_mean.values, label="Mean Forecast", color="#2E6B9E", lw=1.8)
plt.plot(fc_naive.index, fc_naive.values, label="Naïve Forecast", color="#C99700", lw=1.8)
plt.plot(fc_snaive.index, fc_snaive.values, label="Seasonal Naïve (m=4)", color="#107C41", lw=2.2)
plt.plot(fc_drift.index, fc_drift.values, label="Drift Forecast", color="#D83B01", lw=1.8)

plt.title("Australian Quarterly Beer Production: Benchmark Showdown (Replicating FPP3 Fig 5.7)")
plt.xlabel("Quarter")
plt.ylabel("Megalitres")
plt.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.show()


---
## Part 2: Fitted Values, Residuals & Multi-Step Standard Errors (20 Minutes)

### Theoretical Foundation (FPP3 §5.3–§5.5)
- **One-Step In-Sample Residuals**: $e_t = y_t - \hat{y}_{t|t-1}$.
  - For Mean: $\hat{y}_t = \bar{y}$ (0 NaNs).
  - For Naïve: $\hat{y}_t = y_{t-1}$ (first residual is undefined, $M=1$ NaN).
  - For Seasonal Naïve: $\hat{y}_t = y_{t-m}$ (first $m$ residuals undefined, $M=m$ NaNs).
  - For Drift: $\hat{y}_t = y_{t-1} + \hat{c}$ ($M=1$ NaN).
- **Degrees of Freedom**: $\hat{\sigma} = \sqrt{\frac{1}{T - K - M}\sum_{t=1}^T e_t^2}$, where $K$ is the number of estimated parameters ($K=0$ for Naïve/SNaïve, $K=1$ for Mean/Drift).
- **Table 5.2 Multi-Step Scaling**:
  $$\sigma_h = \begin{cases} \hat{\sigma}\sqrt{1 + 1/T} & \text{Mean} \\ \hat{\sigma}\sqrt{h} & \text{Naïve} \\ \hat{\sigma}\sqrt{k+1} & \text{Seasonal Naïve} \\ \hat{\sigma}\sqrt{h(1 + h/(T-1))} & \text{Drift} \end{cases}$$


In [ ]:
def compute_benchmark_residuals(
    y_train: pd.Series,
    method: str,
    m: Optional[int] = None
) -> pd.Series:
    """Computes one-step in-sample residuals e_t = y_t - y_hat_{t|t-1}."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    # idea: for EVERY past time point t ask "what would the method have guessed for t, using only the data before t?"
    # residual = actual value - that guess -> shows how wrong the method was in the past

    method = method.lower()

    if method == "mean":
        fitted = pd.Series(y_train.mean(), index=y_train.index)
        # y_hat_t = y_bar for EVERY t -> the same number in every row
        # Note: y_bar is computed with the WHOLE sample (also the "future" points of t) -> parameter leakage (see markdown)
        # side effect: mean residuals add up to exactly 0 (autograder checks this!)

    elif method == "naive":
        fitted = y_train.shift(1)
        # shift(1) -> moves every value 1 row DOWN -> in row t we now see y_{t-1}
        # e.g. [10, 15, 25] -> [NaN, 10, 15]
        # first row has nothing before it -> NaN (autograder checks: exactly 1 NaN!)

    elif method == "snaive":
        if m is None:
            raise ValueError("snaive needs the seasonal period m (e.g. m=12 for monthly, m=4 for quarterly)")
        fitted = y_train.shift(m)
        # shift(m) -> in row t we see y_{t-m} = the same season one cycle ago (e.g. last year's Q1)
        # the first m rows have no "last cycle" -> m NaNs

    elif method == "drift":
        y = y_train.dropna()
        slope = (y.iloc[-1] - y.iloc[0]) / (len(y) - 1)
        # same slope as in forecast_drift
        fitted = y_train.shift(1) + slope
        # y_hat_t = y_{t-1} + slope -> last value + the average step
        # -> 1 NaN at the start, same as naive

    else:
        raise ValueError(f"Unknown method '{method}' -> use 'mean', 'naive', 'snaive' or 'drift'")

    residuals = y_train - fitted
    # e_t = y_t - y_hat_t -> the actual value minus the guess
    # pandas lines the two series up by DATE automatically, and NaN - anything = NaN -> the NaNs at the start stay

    residuals.name = f"{method}_residuals"
    # f"..." = f-string -> {method} gets replaced by the value of the variable, e.g. "naive_residuals"

    return residuals

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


def compute_forecast_sd(
    residuals: pd.Series,
    method: str,
    h: int,
    T: int,
    m: Optional[int] = None
) -> float:
    """Computes multi-step standard error sigma_h using exact FPP3 Table 5.2 formulas."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    method = method.lower()
    n_params = {"mean": 1, "naive": 0, "snaive": 0, "drift": 1}
    # K = how many parameters each method has to ESTIMATE from the data (needed for the degrees of freedom)
    # mean -> 1 (the average y_bar), naive -> 0 (just copies), snaive -> 0 (just copies), drift -> 1 (the slope)
    # this is a dictionary -> look up a value by its name, e.g. n_params["drift"] gives 1

    if method not in n_params:
        raise ValueError(f"Unknown method '{method}' -> use 'mean', 'naive', 'snaive' or 'drift'")

    # ----- step 1: sigma_hat = how big the typical ONE-step error is -----
    e = residuals.dropna()
    # throw away the NaNs at the start -> what is left are T - M real residuals
    K = n_params[method]

    sigma_hat = np.sqrt(np.sum(e ** 2) / (len(e) - K))
    # sigma_hat = sqrt( sum(e_t^2) / (T - K - M) )
    # len(e) already IS T - M (the NaNs are gone) -> we only subtract K here
    # e ** 2 -> square every residual (** = "to the power of"), so negative and positive errors dont cancel out
    # dividing by a smaller number when we estimated parameters -> slightly bigger sigma = honest correction

    # ----- step 2: scale it up for h steps ahead (Table 5.2) -----
    if method == "mean":
        factor = np.sqrt(1 + 1 / T)
        # the same for every h -> the mean doesnt get more uncertain further out
        # (the extra 1/T = uncertainty of estimating y_bar itself)

    elif method == "naive":
        factor = np.sqrt(h)
        # random walk: every step adds a new independent error -> variance grows like h, sd like sqrt(h)
        # e.g. h = 4 -> sd is 2x the 1-step sd (autograder checks this!)

    elif method == "snaive":
        if m is None:
            raise ValueError("snaive needs the seasonal period m")
        k = (h - 1) // m
        # // = integer division (cuts off the decimals): 3 // 4 = 0, 4 // 4 = 1
        # k = how many FULL seasonal cycles lie between now and step h
        # m = 4: h = 1..4 -> k = 0, h = 5..8 -> k = 1, ...
        factor = np.sqrt(k + 1)
        # -> sd stays flat inside one cycle and JUMPS each time we start a new cycle (stairs)

    elif method == "drift":
        factor = np.sqrt(h * (1 + h / (T - 1)))
        # like naive (sqrt(h)) + a bit extra bc the slope itself is only an estimate

    return float(sigma_hat * factor)
    # float() -> a normal python number instead of a numpy number

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


def compute_prediction_intervals(
    y_hat: pd.Series,
    residuals: pd.Series,
    method: str,
    levels: List[int] = [80, 95],
    m: Optional[int] = None
) -> pd.DataFrame:
    """Constructs analytical Gaussian prediction intervals across horizons h=1..len(y_hat)."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    T = len(residuals)
    # the residual series has one row per training observation (NaNs included) -> its length = T

    horizons = range(1, len(y_hat) + 1)
    # h = 1, 2, ..., (number of forecasts)

    sds = np.array([compute_forecast_sd(residuals, method, h=h, T=T, m=m) for h in horizons])
    # list comprehension = a short for-loop that builds a list: [do_something(h) for h in horizons]
    # -> one sigma_h for every forecast step

    df = pd.DataFrame({"point_forecast": y_hat.values}, index=y_hat.index)
    # start the result table: the dates of the forecast as index + the point forecast as first col

    for level in levels:
        z = stats.norm.ppf(0.5 + level / 200)
        # z = how many sds away from the middle we have to go to catch level% of a normal distribution
        # 95 -> 0.5 + 0.475 = 0.975 -> z = 1.96 (2.5% above +z, 2.5% below -z -> 95% in between)
        # 80 -> 0.5 + 0.40  = 0.90  -> z = 1.28

        df[f"lower_{level}"] = y_hat.values - z * sds
        df[f"upper_{level}"] = y_hat.values + z * sds
        # interval = point forecast +- z * sigma_h
        # numpy does this for every row at once (vectorized)
        # col names like "lower_95" / "upper_80" (autograder checks these!)

    df["sd"] = sds
    # extra col -> so we can see sigma_h growing with h

    return df

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


---
## Part 3: Distribution-Free Bootstrapped Prediction Intervals (20 Minutes)

### Theoretical Foundation (FPP3 §5.5)
When normality fails (e.g. fat-tailed financial shocks or skewed demand), Gaussian analytical intervals severely under-cover true outcomes. Instead of assuming $e_t \sim N(0, \sigma^2)$, we sample historical residuals with replacement to simulate $B = 1,000$ sequential future sample paths:
$$y^*_{T+1} = y_T + e^*_{T+1}, \quad y^*_{T+2} = y^*_{T+1} + e^*_{T+2}, \quad \dots$$
Empirical percentiles (e.g. 2.5% and 97.5%) form the prediction interval bounds without any parametric assumptions.


In [ ]:
def bootstrap_prediction_intervals(
    y_train: pd.Series,
    method: str,
    h: int,
    B: int = 1000,
    levels: List[int] = [80, 95],
    m: Optional[int] = None,
    random_state: int = 42
) -> pd.DataFrame:
    """Simulates B future sample paths and computes empirical quantile prediction intervals."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    # idea: we dont ASSUME the errors are normal -> we reuse the REAL past errors as "possible future shocks"
    # simulate B different possible futures -> look how spread out they are -> that spread = the interval

    method = method.lower()
    if method not in ("mean", "naive", "snaive", "drift"):
        raise ValueError(f"Unknown method '{method}' -> use 'mean', 'naive', 'snaive' or 'drift'")
    if method == "snaive" and m is None:
        raise ValueError("snaive needs the seasonal period m")

    y = y_train.dropna()
    T = len(y)

    # ----- step 1: the "bag" of past errors -----
    residuals = compute_benchmark_residuals(y, method, m=m).dropna().values
    # the historical one-step errors e_t (from Part 2), NaNs removed, as a plain numpy array

    # ----- step 2: draw random shocks from that bag -----
    rng = np.random.default_rng(random_state)
    # random number generator with a fixed seed -> same "random" result every time we run it (RULES.md §3)

    shocks = rng.choice(residuals, size=(B, h), replace=True)
    # pick B * h residuals at random WITH replacement (= the same residual can be picked more than once)
    # shape (B, h) = a table: one ROW per simulated future, one COL per forecast step

    # ----- step 3: build the B future paths step by step -----
    sims = np.empty((B, T + h))
    sims[:, :T] = y.values
    # big table with B rows: the first T cols = the real history (identical in every row), last h cols = still empty
    # [:, :T] -> ":" = all rows, ":T" = cols 0 up to T-1

    y_bar = y.mean()
    slope = (y.iloc[-1] - y.iloc[0]) / (T - 1)

    for j in range(h):
        t = T + j
        # t = the col we fill now (the first future step is col T)

        if method == "mean":
            one_step = y_bar
            # mean: every future value = y_bar + shock
        elif method == "naive":
            one_step = sims[:, t - 1]
            # naive: previous SIMULATED value + shock -> y*_{T+1} = y_T + e*, y*_{T+2} = y*_{T+1} + e*, ...
        elif method == "snaive":
            one_step = sims[:, t - m]
            # snaive: value m steps back (same season) + shock
        else:
            one_step = sims[:, t - 1] + slope
            # drift: previous simulated value + slope + shock

        sims[:, t] = one_step + shocks[:, j]
        # fill col t for ALL B paths at once (vectorized over the rows)
        # the for-loop is only over the h steps bc step j+1 NEEDS the result of step j (that can't be vectorized)

    paths = sims[:, T:]
    # keep only the simulated future part -> shape (B, h)

    # ----- step 4: read the interval off the simulated paths -----
    if method == "mean":
        point = forecast_mean(y_train, h)
    elif method == "naive":
        point = forecast_naive(y_train, h)
    elif method == "snaive":
        point = forecast_snaive(y_train, h, m)
    else:
        point = forecast_drift(y_train, h)
    # the normal point forecast of the method (Part 1) -> also gives us the future dates

    df = pd.DataFrame({"point_forecast": point.values}, index=point.index)

    for level in levels:
        tail = (100 - level) / 2
        # 95 -> 2.5% cut off at the bottom and 2.5% at the top
        df[f"lower_{level}"] = np.percentile(paths, tail, axis=0)
        df[f"upper_{level}"] = np.percentile(paths, 100 - tail, axis=0)
        # np.percentile(..., 2.5, axis=0) -> for every forecast step (col) the value that 2.5% of the paths are below
        # axis=0 = go DOWN the rows (over the B paths) -> one number per col (per step h)

    return df

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


---
## Part 4: Automated Residual Diagnostics Suite (`check_residuals`) (25 Minutes)

### Theoretical Foundation (FPP3 §5.4)
Residual diagnostics audit whether a model has left predictable signal unexploited:
- **Unbiasedness ($E[e_t] = 0$)**: Evaluated via a single-sample t-test ($H_0: \mu = 0$).
- **Portmanteau Ljung-Box Test**: Evaluates whether the first $\ell$ autocorrelations differ jointly from zero:
  $$Q^* = T(T+2)\sum_{k=1}^\ell (T-k)^{-1} r_k^2 \quad \sim \quad \chi^2_{\ell - K}$$
  Degrees of freedom must be adjusted for estimated parameters ($DoF = \ell - K$).
- **Normality**: Evaluated via skewness, kurtosis, and the Jarque-Bera test.


In [ ]:
def check_residuals(
    residuals: pd.Series,
    lags: Optional[int] = None,
    seasonal_m: Optional[int] = None,
    K: int = 0
) -> Dict[str, Any]:
    """
    Executes an automated four-stage residual forensic diagnostic battery.
    Replicates FPP3 Section 5.4 and gg_tsresiduals.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    # good residuals = pure random noise -> the model has used ALL the predictable information
    # 4 checks, every one answers a yes/no question with a statistical test
    # rule for all tests: p-value > 0.05 -> no evidence of a problem -> check passed

    e = residuals.dropna().astype(float)
    T = len(e)
    # remove the NaNs at the start (naive has 1, snaive has m) -> T = number of usable residuals

    if lags is None:
        lags = 2 * seasonal_m if seasonal_m else 10
        lags = max(1, min(lags, T // 5))
    # FPP3 rule of thumb: l = 10 for non-seasonal data, l = 2m for seasonal data, but never more than T/5
    # "a if condition else b" = one-line if/else

    # ----- stage 1: unbiasedness -> is the average error 0? -----
    mean_e = float(e.mean())
    t_stat, t_pvalue = stats.ttest_1samp(e, popmean=0.0)
    # one-sample t-test: H0 = "the true mean of the errors is 0"
    # small p-value -> the errors are systematically too high or too low -> forecasts are biased
    is_unbiased = bool(t_pvalue > 0.05)
    # bool() -> real python True/False (numpy gives np.bool_ and the autograder checks "is True")

    # ----- stage 2: white noise -> is there any pattern left over time? -----
    lb = acorr_ljungbox(e, lags=[lags], model_df=K, return_df=True)
    # Ljung-Box test: looks at the first l autocorrelations TOGETHER
    # H0 = "all of them are 0" = no pattern left = white noise
    # model_df=K -> degrees of freedom = l - K (we lose one per estimated parameter)
    lb_stat = float(lb["lb_stat"].iloc[0])
    lb_pvalue = float(lb["lb_pvalue"].iloc[0])
    # lb is a small table with one row (for our l) -> .iloc[0] = take that row
    is_white_noise = bool(lb_pvalue > 0.05)

    from statsmodels.tsa.stattools import acf
    # acf = the "textbook" autocorrelation function (same formula the Ljung-Box test uses)
    acf_vals = acf(e, nlags=lags, fft=False)[1:]
    # autocorrelation for lag 1..l (lag 0 is always 1 -> [1:] drops it)
    # = how strongly e_t is related to e_{t-1}, e_{t-2}, ...
    bartlett_bound = 1.96 / np.sqrt(T)
    # +- 1.96 / sqrt(T) = the blue dashed lines in the ACF plot -> spikes outside = suspicious
    n_acf_spikes = int(np.sum(np.abs(acf_vals) > bartlett_bound))
    # count how many spikes are outside the bounds (True counts as 1, False as 0)
    # Note: with 20 lags about 1 spike outside is normal just by chance (5%)

    # ----- stage 3: homoscedasticity -> is the variance constant over time? -----
    half = T // 2
    levene_stat, levene_pvalue = stats.levene(e.iloc[:half], e.iloc[half:])
    # simple version: compare the spread of the FIRST half with the SECOND half of the residuals
    # H0 = "both halves have the same variance" -> small p-value = the noise got bigger/smaller ("trumpet" shape)
    is_homoscedastic = bool(levene_pvalue > 0.05)

    # ----- stage 4: normality -> do the errors look like a bell curve? -----
    skewness = float(stats.skew(e))
    # 0 = symmetric, negative = long left tail (big crashes), positive = long right tail
    excess_kurtosis = float(stats.kurtosis(e))
    # 0 = like a normal distribution, big positive = fat tails = more extreme shocks than normal
    jb_stat, jb_pvalue = stats.jarque_bera(e)
    # Jarque-Bera test: combines skewness + kurtosis, H0 = "the errors are normal"
    is_normal = bool(jb_pvalue > 0.05)
    # if this fails -> the Gaussian intervals of Part 2 are not trustworthy -> use the bootstrap of Part 3

    return {
        "n_obs": T,
        "lags": int(lags),
        "dof": int(lags - K),
        "mean": mean_e,
        "t_pvalue": float(t_pvalue),
        "is_unbiased": is_unbiased,
        "ljung_box_stat": lb_stat,
        "ljung_box_pvalue": lb_pvalue,
        "is_white_noise": is_white_noise,
        "bartlett_bound": float(bartlett_bound),
        "n_acf_spikes": n_acf_spikes,
        "levene_pvalue": float(levene_pvalue),
        "is_homoscedastic": is_homoscedastic,
        "skewness": skewness,
        "excess_kurtosis": excess_kurtosis,
        "jarque_bera_pvalue": float(jb_pvalue),
        "is_normal": is_normal,
    }
    # dictionary -> one key per result, the demo cell prints them all (autograder checks the main keys!)

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


In [ ]:
# Forensic Demonstration on Google Closing Stock (Replicating FPP3 Fig 5.13)
google = load_and_index_series(str(DATA_DIR / "gafa_stock_google.csv"), "observation_date", "adj_close", freq="B")
res_google = compute_benchmark_residuals(google, method="naive")
diag_google = check_residuals(res_google, lags=10, K=0)

print("--- Google Stock Residual Diagnostics ---")
for k, v in diag_google.items():
    print(f"{k:18s}: {v}")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
# 1. Top Panel: Time Plot
ax_top = plt.subplot2grid((2, 2), (0, 0), colspan=2)
ax_top.plot(res_google.index, res_google.values, color="#1A3A6B", lw=1.2)
ax_top.axhline(0, color="black", ls="--", alpha=0.6)
ax_top.set_title("Google Stock Residuals: Time Plot (Inspecting Variance & Outliers)")
ax_top.set_ylabel("Residual ($)")

# 2. Bottom Left: ACF Plot with Bartlett Bounds
ax_acf = plt.subplot2grid((2, 2), (1, 0))
T_g = len(res_google.dropna())
bartlett = 1.96 / np.sqrt(T_g)
lags_range = range(1, 21)
acf_vals = [res_google.dropna().autocorr(lag=l) for l in lags_range]
ax_acf.vlines(lags_range, [0], acf_vals, color="#1A3A6B", lw=2)
ax_acf.scatter(lags_range, acf_vals, color="#1A3A6B", zorder=3)
ax_acf.axhline(bartlett, color="#2E6B9E", ls="--", label=f"95% Bartlett (±{bartlett:.3f})")
ax_acf.axhline(-bartlett, color="#2E6B9E", ls="--")
ax_acf.axhline(0, color="black", lw=0.8)
ax_acf.set_title("Sample Autocorrelation Function (ACF)")
ax_acf.set_xlabel("Lag")
ax_acf.set_ylabel("ACF")
ax_acf.legend(frameon=True)

# 3. Bottom Right: Histogram & Normal Fit
ax_hist = plt.subplot2grid((2, 2), (1, 1))
sns.histplot(res_google.dropna(), kde=True, ax=ax_hist, color="#1A3A6B", stat="density")
ax_hist.set_title("Residual Distribution vs Gaussian Curve")
ax_hist.set_xlabel("Residual ($)")

plt.tight_layout()
plt.show()


---
## Part 5: Expanding-Window Backtest & Empirical Coverage Auditing (20 Minutes)

### Theoretical Foundation (FPP3 §5.5)
A prediction interval claiming 95% nominal coverage is useless if empirical test coverage is only 80%. We audit calibration across rolling temporal origins:
$$\text{Coverage}_h = \frac{1}{N}\sum_{t=1}^N \mathbb{I}\left(L_{t+h|t} \le y_{t+h} \le U_{t+h|t}\right)$$


In [ ]:
def evaluate_empirical_coverage(
    y_series: pd.Series,
    method: str,
    initial_train_size: int,
    h: int,
    level: int = 95,
    m: Optional[int] = None
) -> float:
    """Conducts an expanding-origin backtest to audit empirical coverage at horizon h."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    # idea: pretend we are back in the past, make a forecast + interval, then check if the REAL value landed inside
    # repeat for many starting points ("origins") -> % of hits = empirical coverage
    # a good 95% interval should catch the real value about 95% of the time

    method = method.lower()
    y = y_series.dropna()
    n = len(y)

    if initial_train_size < 2 or initial_train_size + h > n:
        raise ValueError("initial_train_size too small or too big for this series and horizon h")

    lower_col, upper_col = f"lower_{level}", f"upper_{level}"

    hits = []
    # empty list -> we add True (inside) or False (outside) for every origin

    for origin in range(initial_train_size, n - h + 1):
        # expanding window: origin = how many observations we are allowed to see
        # first round: the first initial_train_size points, then 1 more each round (window grows, never looks ahead)
        # last origin = n - h -> the value h steps later is still inside the data

        train = y.iloc[:origin]
        # ONLY the past up to the origin -> no data leakage (RULES.md §2)
        actual = y.iloc[origin + h - 1]
        # the real value h steps after the origin (the one we try to catch)

        if method == "mean":
            fc = forecast_mean(train, h)
        elif method == "naive":
            fc = forecast_naive(train, h)
        elif method == "snaive":
            fc = forecast_snaive(train, h, m)
        elif method == "drift":
            fc = forecast_drift(train, h)
        else:
            raise ValueError(f"Unknown method '{method}' -> use 'mean', 'naive', 'snaive' or 'drift'")
        res = compute_benchmark_residuals(train, method, m=m)
        pi = compute_prediction_intervals(fc, res, method, levels=[level], m=m)
        # forecast + residuals + interval with the functions from Part 1 + 2, fitted on train ONLY

        lower = pi[lower_col].iloc[h - 1]
        upper = pi[upper_col].iloc[h - 1]
        # only the h-th step matters here (iloc[h - 1] bc python counts from 0)

        hits.append(bool(lower <= actual <= upper))
        # .append -> adds one item at the end of the list
        # lower <= actual <= upper -> True if the real value is inside the interval

    return float(np.mean(hits))
    # mean of True/False = share of True (True = 1, False = 0) -> e.g. 0.93 = 93% coverage

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


---
## Part 6: Decomposition Forecasting (STLF Pipeline) (15 Minutes)

### Theoretical Foundation (FPP3 §5.7)
Decomposition turns a difficult non-stationary seasonal series into two simple forecasting problems:
1. Decompose $y_t = S_t + A_t$ via STL (where $A_t = T_t + R_t$ is seasonally adjusted).
2. Project seasonal component $\hat{S}_{T+h}$ using `forecast_snaive`.
3. Project seasonally adjusted component $\hat{A}_{T+h}$ using `forecast_drift` or `forecast_naive`.
4. Reseasonalise by summing the components: $\hat{y}_{T+h} = \hat{S}_{T+h} + \hat{A}_{T+h}$.


In [ ]:
def forecast_stlf(
    y_train: pd.Series,
    h: int,
    m: int,
    method_trend: str = "drift"
) -> pd.Series:
    """Forecasting with decomposition (STLF)."""
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    # same idea as Task 6 in workshop_02, but ADDITIVE (y = S + A) and built from our Part 1 forecasters

    y = y_train.dropna()
    # STL cant handle NaNs -> drop them first

    stl_res = STL(y, period=m, robust=True).fit()
    # step 1: split the series into trend + seasonal + remainder (additive STL)
    # period=m -> season length (4 for quarterly, 12 for monthly)
    # robust=True -> outliers (e.g. COVID) get downweighted so they dont bend the seasonal pattern

    seasonal = stl_res.seasonal
    seasonally_adjusted = y - seasonal
    # A_t = y_t - S_t = trend + remainder -> the series with the seasonal ups and downs removed

    seasonal_fc = forecast_snaive(seasonal, h=h, m=m)
    # step 2: seasonal part -> seasonal naive = repeat the last cycle of S_t

    method_trend = method_trend.lower()
    if method_trend == "drift":
        sa_fc = forecast_drift(seasonally_adjusted, h=h)
    elif method_trend == "naive":
        sa_fc = forecast_naive(seasonally_adjusted, h=h)
    else:
        raise ValueError(f"Unknown method_trend '{method_trend}' -> use 'drift' or 'naive'")
    # step 3: seasonally adjusted part -> has no seasonality anymore, so a simple NON-seasonal method works
    # drift = keep the trend going, naive = stay flat at the last level

    forecast = seasonal_fc + sa_fc
    # step 4: reseasonalise -> y_hat = S_hat + A_hat (ADD bc our decomposition is additive)
    # both series have the same future dates -> pandas adds them row by row

    forecast.name = "stlf_forecast"

    return forecast

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstration: STLF on US Monthly Retail Sales (RSAFS)
rsafs = load_and_index_series(str(DATA_DIR / "RSAFS.csv"), "observation_date", "RSAFSNA", freq="MS")
train_r = rsafs.loc["2010-01-01":"2022-12-01"]
test_r = rsafs.loc["2023-01-01":"2024-06-01"]
h_r = len(test_r)

fc_stlf = forecast_stlf(train_r, h=h_r, m=12, method_trend="drift")

plt.figure(figsize=(12, 5))
plt.plot(train_r.index[-36:], train_r.values[-36:], label="Historical RSAFS", color="#1A3A6B", lw=2)
plt.plot(test_r.index, test_r.values, label="Actual Test Data", color="black", lw=2.5, ls="--")
plt.plot(fc_stlf.index, fc_stlf.values, label="STLF Forecast (STL + SNaive + Drift)", color="#107C41", lw=2.2)

plt.title("Forecasting US Retail Sales with Decomposition (STLF Pipeline)")
plt.xlabel("Month")
plt.ylabel("Million USD")
plt.legend(frameon=True)
plt.tight_layout()
plt.show()


---
## Part 7: Practical Syllabus Interview Clinic

#### Question 1: Implement a seasonal naive forecast in a few lines of Python.
**Model Answer:**  
Extract the final $m$ observations of the training series (`y_train.iloc[-m:].values`) and project them forward using modulo arithmetic: `[last_season[(i - 1) % m] for i in range(1, h + 1)]`. Wrap in a `pd.Series` with a properly constructed future `pd.DatetimeIndex`. This requires zero parameter estimation and forms the non-negotiable benchmark for all seasonal economic series.

#### Question 2: Which residual plots and tests do you check, and what does each failure look like?
**Model Answer:**  
We check the three-panel diagnostic battery (time plot, ACF, histogram) and the Ljung-Box portmanteau test:
1. **Time Plot**: Inspect for non-constant variance (heteroskedastic funnels) or abrupt shocks. A failure looks like an expanding trumpet shape.
2. **ACF Plot**: Inspect for autocorrelations outside Bartlett bounds ($\pm 1.96/\sqrt{T}$). A failure looks like significant spikes at lag 1 (momentum) or seasonal lags (unmodeled seasonality).
3. **Histogram**: Inspect for symmetry and bell-curve decay. A failure looks like severe skewness or heavy fat tails.
4. **Ljung-Box Test**: Evaluates joint autocorrelation up to lag $\ell$. If $p < 0.05$, residuals contain unexploited information and fail the white noise requirement.

#### Question 3: How do you obtain prediction intervals from a benchmark and check their coverage?
**Model Answer:**  
- **Analytical Intervals**: Compute residual variance $\hat{\sigma}^2 = \frac{1}{T - K - M}\sum e_t^2$. Scale $\hat{\sigma}$ across horizons using Table 5.2 formulas (e.g. $\hat{\sigma}\sqrt{h}$ for Naïve, $\hat{\sigma}\sqrt{k+1}$ for Seasonal Naïve). Construct bounds as $\hat{y} \pm z \sigma_h$.
- **Bootstrapped Intervals**: When normality fails, sample residuals with replacement sequentially over $B=1,000$ paths and compute empirical percentiles (2.5% and 97.5%).
- **Coverage Audit**: Conduct an expanding-origin backtest. Count the fraction of test actuals falling inside $[L, U]$. If nominal 95% intervals capture $\ge 95\%$ of outcomes, calibration is verified.
